# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YomnaImad07/FlyRank-ML-Internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [8]:
%pip -q install duckdb huggingface_hub

import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

span = con.sql(f"""
    SELECT MIN(report_date) AS min_d, MAX(report_date) AS max_d,
           DATEDIFF('day', MIN(report_date), MAX(report_date)) AS total_days
    FROM {TABLES['fact_daily']}
""").df()
half_window = span['total_days'].iloc[0] // 2

FINAL_THRESHOLD = 100

features = con.sql(f"""
    WITH bounds AS (SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL {half_window} DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL {half_window} DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL {half_window} DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL {half_window} DAY THEN f.gsc_avg_position END)       AS pos_last30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL {half_window * 2} DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= {FINAL_THRESHOLD}
    )
    SELECT * FROM windowed
""").df()

qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)     AS visible_queries,
           ANY_VALUE(rare_impressions_share)          AS rare_share,
           ANY_VALUE(anonymized_impressions_share)    AS anon_share,
           MAX(impressions_90d)                       AS top_query_impressions,
           SUM(impressions_90d)                       AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

data = features.merge(qsignals, on='content_hash_id', how='left')

volatility = con.sql(f"""
    WITH bounds AS (SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']})
    SELECT f.content_hash_id, STDDEV(f.gsc_avg_position) AS position_volatility
    FROM {TABLES['fact_daily']} f, bounds b
    WHERE f.report_date > b.end_d - INTERVAL {half_window * 2} DAY
    GROUP BY 1
""").df()
data = data.merge(volatility, on='content_hash_id', how='left')

data['top_query_share'] = data['top_query_impressions'] / data['kept_impressions']

fill_cols = ['visible_queries', 'rare_share', 'anon_share', 'top_query_share', 'position_volatility']
data[fill_cols] = data[fill_cols].fillna(0)

data['is_declining'] = (data['imp_last30'] < 0.8 * data['imp_prev30']).astype(int)

feature_cols = ['imp_prev30', 'visible_queries', 'rare_share',
                 'anon_share', 'top_query_share', 'position_volatility']

# --- تدريب الموديل بالـ grouped split (النسخة الأمانة من notebook 09) ---
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

model_data = data.dropna(subset=feature_cols).reset_index(drop=True)
X, y = model_data[feature_cols], model_data['is_declining']
groups = model_data['client_hash_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
X_tr_g, X_te_g = X.iloc[train_idx], X.iloc[test_idx]
y_tr_g, y_te_g = y.iloc[train_idx], y.iloc[test_idx]

clf_after = RandomForestClassifier(n_estimators=200, random_state=42)
clf_after.fit(X_tr_g, y_tr_g)
f1_after = f1_score(y_te_g, clf_after.predict(X_te_g))

# لازم نجيب f1_before كمان عشان نستخدمها في تصدير قسم 5
from sklearn.model_selection import train_test_split
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
clf_before = RandomForestClassifier(n_estimators=200, random_state=42)
clf_before.fit(X_tr, y_tr)
f1_before = f1_score(y_te, clf_before.predict(X_te))

print("data rows:", len(data), "| model_data rows:", len(model_data))
print(f"f1_before: {f1_before:.3f} | f1_after: {f1_after:.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

data rows: 77155 | model_data rows: 77155
f1_before: 0.633 | f1_after: 0.651


In [9]:
import pandas as pd

# استخدمي الموديل المدرّب على الـ grouped split (الأمانة العلمية) من notebook 09
# نطبقه على كل الداتا عشان نطلع الـ ranked queue
model_data['risk_score'] = clf_after.predict_proba(model_data[feature_cols])[:, 1]

# reason codes: اربطي كل صف بأقوى إشارة فعلية اتأكدت من notebook 06
def assign_reason(row):
    reasons = []
    if row['position_volatility'] > data['position_volatility'].quantile(0.75):
        reasons.append("HIGH_VOLATILITY")  # confirmed signal من week 4
    if row['imp_prev30'] > 0 and row['imp_last30'] < 0.6 * row['imp_prev30']:
        reasons.append("SHARP_DROP")
    if not reasons:
        reasons.append("GENERAL_RISK")
    return ", ".join(reasons)

model_data['reason_code'] = model_data.apply(assign_reason, axis=1)

# الـ queue النهائي: أعلى 50 صفحة بترتيب risk_score
action_queue = model_data.sort_values('risk_score', ascending=False)[
    ['content_hash_id', 'client_hash_id', 'risk_score', 'reason_code',
     'position_volatility', 'imp_prev30', 'imp_last30']
].head(50).reset_index(drop=True)

print(action_queue.head(10))
print(f"\nTotal queue size: {len(action_queue)}")
print(f"\nReason code distribution:\n{model_data['reason_code'].value_counts()}")

            content_hash_id           client_hash_id  risk_score  \
0  content_cb3c73858055f7a3  client_fef1a8f436438636         1.0   
1  content_592716072ac3e4f5  client_e5c2aa26a8598242         1.0   
2  content_2ba4331624f107cd  client_73cda7b4e4f265ea         1.0   
3  content_2c6bd85c3e92a1d8  client_62f4a7e64f5e0096         1.0   
4  content_9aa69550164f13d1  client_73cda7b4e4f265ea         1.0   
5  content_39fef85c7566487b  client_e5c2aa26a8598242         1.0   
6  content_9e687814f0b3c86a  client_62f4a7e64f5e0096         1.0   
7  content_b5d2fe44c124b558  client_06d356715a8ff3b6         1.0   
8  content_6cf60bcde0666a60  client_e00b29e582949543         1.0   
9  content_995a4a7254c86f20  client_73cda7b4e4f265ea         1.0   

                   reason_code  position_volatility  imp_prev30  imp_last30  
0  HIGH_VOLATILITY, SHARP_DROP            16.892879       243.0        34.0  
1  HIGH_VOLATILITY, SHARP_DROP            16.689185       246.0       131.0  
2              HI

## 1. Ranked actions + reason codes

The queue ranks pages by the grouped-split model's predicted decline probability,
highest first — the top 50 rows all carry a risk_score of 1.0, meaning the model is
maximally confident on this slice. Each page carries a reason code so a reviewer
knows *why* it's flagged, not just that it is: HIGH_VOLATILITY marks pages in the top
quartile of position_volatility (the one signal that held up under Week 4's audit),
SHARP_DROP marks pages already showing a steep impression drop between the two
windows, and GENERAL_RISK covers pages the model flags without either specific
pattern present. Across the full dataset, most flagged pages (43,886) fall into
GENERAL_RISK, while 7,996 pages show both warning signs at once — those are the
pages a reviewer should look at first, since two independent signals agree. A
reviewer opening row 1 reads: "flagged because its ranking position has been
swinging heavily (volatility ≈ 16.9) and impressions already dropped from 243 to 34
over the observation window" — not a bare probability number.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [14]:

model_scope = {
    "trained_on_days": int(span['total_days'].iloc[0]),
    "min_impressions_threshold": FINAL_THRESHOLD,
    "n_training_clients": model_data['client_hash_id'].nunique(),
}
print(model_scope)

{'trained_on_days': 29, 'min_impressions_threshold': 100, 'n_training_clients': 45}


## 2. Intended use and limits

Intended user: a content team lead triaging which pages to review first, not an
automated action system. Valid range: pages with at least 100 impressions in the
pre-period (the training threshold) and clients similar in scale to those in the
training data — the model has not been tested on brand-new clients with very
different traffic patterns, or on pages below the impression floor. It stops being
valid outside the ~29-day window the underlying sample covers; a longer or seasonal
window could change which signals matter. This is a prioritization aid, not a
diagnosis of *why* a page is declining.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [11]:

low_confidence = action_queue[action_queue['imp_prev30'] < FINAL_THRESHOLD * 1.5]
print(f"Queue rows close to the impression floor (lower-confidence cases): {len(low_confidence)}")

Queue rows close to the impression floor (lower-confidence cases): 7


## 3. Human review + the no-go list

Before acting on any row, a reviewer must check: the page's content type (a
seasonal or time-limited page declining "on schedule" isn't a real risk), whether
a known site-wide event (migration, outage) explains the drop instead of an organic
issue, and pages sitting close to the impression floor, where the signal is
statistically noisier.

**No-go list — never automate:**
- Auto-publishing or auto-editing content based on the risk score alone
- Auto-deprioritizing or removing a page without a human looking at it first
- Treating the risk score as a ranking-cause explanation to report to a client
</br>
The model output is a starting point for a queue, never a standalone trigger for
action on a client's live content.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [12]:

print(data[feature_cols].describe().loc[['mean', 'std']])

       imp_prev30  visible_queries  rare_share  anon_share  top_query_share  \
mean  1323.910479        28.293656    0.095991    0.658656         0.325467   
std   4192.900377        59.734115    0.096537    0.237756         0.234907   

      position_volatility  
mean             7.070607  
std              6.831238  


## 4. Monitoring / retrain triggers

Retrain or re-audit the model if: (1) the mean or spread of position_volatility or
imp_prev30 shifts meaningfully from the values above — a sign the underlying traffic
patterns changed; (2) the queue's precision on manually-reviewed pages drops
noticeably below the audited baseline; or (3) a new data window becomes available
that's long enough to rebuild the last30/prev30 split at the originally intended
30/30-day size instead of the ~14/14-day compromise used here. Any of these would
mean the recommendations are stale and the signal audit (Week 4) and validation
(Week 6) should be re-run before trusting the queue again.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [13]:
import os

os.makedirs('work/outputs', exist_ok=True)


action_queue.to_csv('work/outputs/action_queue.csv', index=False)


import json
signal_summary = {
    "position_volatility": {"verdict": "CONFIRMED", "spread": "37.4% -> 59.8%"},
    "rare_share": {"verdict": "MIXED"},
    "visible_queries": {"verdict": "FALSE"},
    "top_query_share_flag": {"verdict": "FALSE", "flagged_rate": 0.486, "unflagged_rate": 0.501},
    "model_f1_before_random_split": float(f1_before),
    "model_f1_after_grouped_split": float(f1_after),
}
with open('work/outputs/signal_summary.json', 'w') as f:
    json.dump(signal_summary, f, indent=2)

print("Exported:")
print("- work/outputs/action_queue.csv")
print("- work/outputs/signal_summary.json")

Exported:
- work/outputs/action_queue.csv
- work/outputs/signal_summary.json


## 5. Exports for the paper

The ranked queue (top 50 pages with risk scores and reason codes) and a JSON summary
of every audited signal from Weeks 4–6 are written to work/outputs/. These are the
files the capstone paper's Results and Methodology sections will draw numbers and
tables from directly, rather than re-deriving them by hand.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.